# Batch Inference Pipeline

This notebook demonstrates how to run the batch inference pipeline over multiple
reflectometry experiments and how to analyse and visualise the results.

Topics covered:

1. Running a small batch via `BatchInferencePipeline`
2. Loading saved batch results from JSON
3. Summary statistics and MAPE distribution
4. Per-parameter MAPE breakdown plot
5. Identifying edge cases (high-error experiments)
6. Regenerating plots from existing results with `replot_batch_results`

## 1. Setup

In [ ]:
import sys, os, json
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt

from config import DATA_DIRECTORY, BATCH_RESULTS_DIR

print(f"Project root      : {PROJECT_ROOT}")
print(f"Data directory    : {DATA_DIRECTORY}")
print(f"Batch results dir : {BATCH_RESULTS_DIR}")

## 2. Run a Small Batch

`BatchInferencePipeline` orchestrates the full workflow: experiment discovery,
preprocessing, inference, metrics, and result serialisation.

Set `NUM_EXPERIMENTS` to a small number (e.g. 5) for a quick demonstration run.
Set to `None` to process all available experiments in the layer split.

In [ ]:
from batch_pipeline import BatchInferencePipeline

NUM_EXPERIMENTS  = 5      # set to None to process all
LAYER_COUNT      = 1
PRIORS_DEV       = 30     # % of true value
SLD_MODE         = "none" # "none" | "backing" | "all"
PROMINENT        = False  # True = restrict to prominent-feature subset

pipeline = BatchInferencePipeline(
    num_experiments=NUM_EXPERIMENTS,
    layer_count=LAYER_COUNT,
    data_directory=DATA_DIRECTORY,
    use_narrow_priors=True,
    narrow_priors_deviation=PRIORS_DEV / 100.0,
    fix_sld_mode=SLD_MODE,
    use_prominent_features=PROMINENT,
)

batch_results, output_dir = pipeline.run_batch_inference()
print(f"\nResults saved to: {output_dir}")

## 3. Load Results from an Existing Batch

To work with a previously completed run without re-running inference, load its `batch_results.json` directly.
Update `EXISTING_BATCH_DIR` to point at any directory under `batch_inference_results/`.

In [ ]:
# Point this at any completed batch directory
EXISTING_BATCH_DIR = Path(PROJECT_ROOT) / BATCH_RESULTS_DIR / "075_3169exps_1layers_99constraint_28october2025_15_08"

results_file = EXISTING_BATCH_DIR / "batch_results.json"

with open(results_file) as f:
    loaded_results = json.load(f)

successful = {k: v for k, v in loaded_results.items() if v.get("success", False)}
print(f"Loaded {len(loaded_results)} total results, {len(successful)} successful")

## 4. Summary Statistics

`batch_analysis.create_summary_statistics` aggregates per-experiment metrics into dataset-level statistics.

In [ ]:
from batch_analysis import create_summary_statistics, print_summary_statistics, print_mape_distribution

summary = create_summary_statistics(successful, layer_count=LAYER_COUNT)
print_summary_statistics(summary)
print()
print_mape_distribution(successful)

## 5. MAPE Distribution and Parameter Breakdown Plots

`create_batch_analysis_plots` produces all standard batch plots in one call.
Set `save=True` to write PDFs to `output_dir`.

In [ ]:
from plotting_utils import create_batch_analysis_plots

plot_paths = create_batch_analysis_plots(
    successful,
    layer_count=LAYER_COUNT,
    output_dir=EXISTING_BATCH_DIR / "plots_nb",
    save=True,
)

for name, path in plot_paths.items():
    if path is not None:
        print(f"  {name}: {path}")

## 6. Detect Edge Cases

`detect_edge_cases` flags experiments where the model produced unusually large errors.
These are worth investigating manually to check whether they represent genuine failures or
pathological structures that lie outside the training distribution.

In [ ]:
from batch_analysis import detect_edge_cases

edge_cases = detect_edge_cases(successful)
print(f"Edge cases detected: {len(edge_cases)}")
for exp_id, info in list(edge_cases.items())[:5]:
    print(f"  Experiment {exp_id}: overall MAPE = {info.get('overall_mape', 'N/A'):.1f}%")

## 7. Re-Generate Plots from CLI

Plots can be regenerated from any completed batch directory without re-running inference:

```bash
python replot_batch_results.py --results-dir batch_inference_results/<batch_dir>
```

Or using the Python API:

In [ ]:
from replot_batch_results import replot_batch_results

saved_plots = replot_batch_results(
    results_dir=str(EXISTING_BATCH_DIR),
    output_dir=str(EXISTING_BATCH_DIR / "replot_nb"),
)
print(f"Regenerated {len(saved_plots)} plots.")